# 07 · Optimización intensiva V5 · Cinco modelos

Este notebook continúa desde **V5 (192 características)** y realiza una búsqueda
de hiperparámetros mucho más seria para los cinco modelos del proyecto.

## Diseño

**Fase A**
- Regresión Logística: 12 configuraciones
- Random Forest: 30 configuraciones
- XGBoost: 30 configuraciones
- SVM RBF: 20 configuraciones
- MLP: 20 configuraciones

Cada configuración se evalúa en los **4 folds temporales expansivos**.

**Fase B**
- Se seleccionan automáticamente los **2 mejores modelos ML** de la Fase A.
- Se construye un espacio de búsqueda local alrededor de sus mejores parámetros.
- Se prueban 25 configuraciones adicionales por modelo.

La climatología V5 se vuelve a calcular dentro de cada fold usando solo el
subentrenamiento disponible en ese momento.

> **No se utiliza 2018–2021, prueba temporal 2022–2025 ni holdout espacial.**

## 0. Dependencias

In [1]:
%pip install -q scikit-learn xgboost joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports

In [2]:
from pathlib import Path
import importlib
import json
import sys
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import ParameterSampler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Imports listos.")

Imports listos.


## 2. Localizar el proyecto

In [3]:
candidates = [
    Path.cwd(),
    Path.cwd() / "rain-threat-classifier",
    Path("/content/rain-threat-classifier"),
    Path("/content/drive/MyDrive/rain-threat-classifier"),
]

PROJECT_DIR = next(
    (
        p for p in candidates
        if (p / "resultados_completo" / "dataset_modelo_mensual_v3.csv").exists()
        and (p / "resultados_completo" / "columnas_modelo_v3.txt").exists()
        and (p / "resultados_completo" / "indicadores_mensuales_todas_zonas.csv").exists()
        and (p / "04_climatologia_era5land.py").exists()
    ),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "No se encontró el proyecto. Deben existir dataset V3, columnas V3, "
        "indicadores mensuales y 04_climatologia_era5land.py."
    )

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

clim = importlib.import_module("04_climatologia_era5land")
clim = importlib.reload(clim)

DATA_DIR = PROJECT_DIR / "resultados_completo"
OUTPUT_DIR = (
    PROJECT_DIR
    / "resultados_experimentos"
    / "v5_optimizacion_intensiva"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

PROJECT_DIR: c:\Users\Kathy\Documents\GitHub\rain-threat-classifier
OUTPUT_DIR: c:\Users\Kathy\Documents\GitHub\rain-threat-classifier\resultados_experimentos\v5_optimizacion_intensiva


## 3. Carga de datos

In [4]:
DATASET_PATH = DATA_DIR / "dataset_modelo_mensual_v3.csv"
FEATURES_PATH = DATA_DIR / "columnas_modelo_v3.txt"
MONTHLY_PATH = DATA_DIR / "indicadores_mensuales_todas_zonas.csv"

df = pd.read_csv(DATASET_PATH)
df["period_start"] = pd.to_datetime(df["period_start"])
df["target_period_start"] = pd.to_datetime(df["target_period_start"])

monthly = pd.read_csv(MONTHLY_PATH)
monthly["period_start"] = pd.to_datetime(monthly["period_start"])

features_v3 = [
    line.strip()
    for line in FEATURES_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

features_v5 = features_v3 + clim.CLIMATE_FEATURE_NAMES

assert len(features_v3) == 167
assert len(clim.CLIMATE_FEATURE_NAMES) == 25
assert len(features_v5) == 192

FORBIDDEN_FEATURES = {
    "target_amenaza",
    "target_rx5day_mm",
}

assert not (FORBIDDEN_FEATURES & set(features_v5))

print("Dataset:", df.shape)
print("Features V3:", len(features_v3))
print("Features climatológicas:", len(clim.CLIMATE_FEATURE_NAMES))
print("Features V5:", len(features_v5))

Dataset: (6120, 209)
Features V3: 167
Features climatológicas: 25
Features V5: 192


## 4. Entrenamiento y clases

In [5]:
train_df = df[df["split"] == "entrenamiento"].copy()

label_encoder = LabelEncoder()
label_encoder.fit(train_df["target_amenaza"])
CLASS_NAMES = list(label_encoder.classes_)

print("Filas entrenamiento:", len(train_df))
print("Zonas:", train_df["zone_id"].nunique())
print("Clases:", dict(enumerate(CLASS_NAMES)))
print(
    "Target:",
    train_df["target_period_start"].min().date(),
    "→",
    train_df["target_period_start"].max().date(),
)

Filas entrenamiento: 3744
Zonas: 12
Clases: {0: 'Alta', 1: 'Baja', 2: 'Media'}
Target: 1992-01-01 → 2017-12-01


## 5. Folds temporales y climatología V5

In [6]:
TEMPORAL_FOLDS = [
    ("F1", "2004-12-01", "2005-01-01", "2007-12-01"),
    ("F2", "2007-12-01", "2008-01-01", "2010-12-01"),
    ("F3", "2010-12-01", "2011-01-01", "2013-12-01"),
    ("F4", "2013-12-01", "2014-01-01", "2017-12-01"),
]

fold_data = []
fold_summary = []

for name, train_end, val_start, val_end in TEMPORAL_FOLDS:
    subtrain = train_df[
        train_df["target_period_start"] <= pd.Timestamp(train_end)
    ].copy()

    subval = train_df[
        train_df["target_period_start"].between(
            pd.Timestamp(val_start),
            pd.Timestamp(val_end),
        )
    ].copy()

    cutoff = subtrain["period_start"].max()
    zones = subtrain["zone_id"].unique()

    reference = clim.fit_climatology(
        monthly=monthly,
        cutoff=cutoff,
        zones=zones,
    )

    X_train = clim.add_climate_features(
        frame=subtrain,
        base_X=subtrain[features_v3].copy(),
        reference=reference,
    )
    X_val = clim.add_climate_features(
        frame=subval,
        base_X=subval[features_v3].copy(),
        reference=reference,
    )

    y_train = label_encoder.transform(subtrain["target_amenaza"])
    y_val = label_encoder.transform(subval["target_amenaza"])

    assert X_train.shape[1] == 192
    assert X_val.shape[1] == 192
    assert X_train.isna().sum().sum() == 0
    assert X_val.isna().sum().sum() == 0

    fold_data.append({
        "name": name,
        "train_frame": subtrain,
        "val_frame": subval,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "cutoff": cutoff,
    })

    fold_summary.append({
        "fold": name,
        "train_filas": len(subtrain),
        "val_filas": len(subval),
        "climatologia_hasta": cutoff.date(),
        "val_target_desde": pd.Timestamp(val_start).date(),
        "val_target_hasta": pd.Timestamp(val_end).date(),
    })

display(pd.DataFrame(fold_summary))

,fold,train_filas,val_filas,climatologia_hasta,val_target_desde,val_target_hasta
0,F1,1872,432,2004-11-01,2005-01-01,2007-12-01
1,F2,2304,432,2007-11-01,2008-01-01,2010-12-01
2,F3,2736,432,2010-11-01,2011-01-01,2013-12-01
3,F4,3168,576,2013-11-01,2014-01-01,2017-12-01


## 6. Métricas

In [7]:
def calculate_metrics(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=np.arange(len(CLASS_NAMES)),
        zero_division=0,
    )

    result = {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
    }

    for i, class_name in enumerate(CLASS_NAMES):
        key = class_name.lower()
        result[f"precision_{key}"] = float(precision[i])
        result[f"recall_{key}"] = float(recall[i])
        result[f"f1_{key}"] = float(f1[i])
        result[f"support_{key}"] = int(support[i])

    return result

## 7. Baselines

In [8]:
baseline_rows = []

for fold in fold_data:
    dummy = DummyClassifier(strategy="most_frequent")
    dummy.fit(fold["X_train"], fold["y_train"])
    dummy_pred = dummy.predict(fold["X_val"])

    persistence_pred = label_encoder.transform(
        fold["val_frame"]["amenaza_mes"]
    )

    baseline_rows.append({
        "modelo": "Dummy",
        "fold": fold["name"],
        **calculate_metrics(fold["y_val"], dummy_pred),
    })

    baseline_rows.append({
        "modelo": "Persistencia",
        "fold": fold["name"],
        **calculate_metrics(fold["y_val"], persistence_pred),
    })

baseline_fold_table = pd.DataFrame(baseline_rows)

baseline_cv = (
    baseline_fold_table
    .groupby("modelo", as_index=False)
    .agg(
        macro_f1_cv_mean=("macro_f1", "mean"),
        macro_f1_cv_std=("macro_f1", "std"),
        balanced_accuracy_cv_mean=("balanced_accuracy", "mean"),
        recall_alta_cv_mean=("recall_alta", "mean"),
    )
    .sort_values("macro_f1_cv_mean", ascending=False)
)

display(baseline_cv)

,modelo,macro_f1_cv_mean,macro_f1_cv_std,balanced_accuracy_cv_mean,recall_alta_cv_mean
1,Persistencia,0.369561,0.031969,0.369602,0.349616
0,Dummy,0.155065,0.016693,0.333333,0.250000


## 8. Modelos

Durante la búsqueda se desactiva `probability=True` en SVM porque no necesitamos
probabilidades para calcular Macro F1. Esto hace el tuning bastante más rápido.
La versión final podrá reentrenarse con probabilidades si el SVM resulta ganador.

In [9]:
models = {
    "Regresion_Logistica": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            solver="lbfgs",
            random_state=RANDOM_STATE,
        )),
    ]),

    "Random_Forest": RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),

    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),

    "SVM_RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            probability=False,
            cache_size=2048,
            random_state=RANDOM_STATE,
        )),
    ]),

    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            solver="adam",
            max_iter=800,
            early_stopping=True,
            random_state=RANDOM_STATE,
        )),
    ]),
}

## 9. Espacios amplios de hiperparámetros · Fase A

In [10]:
param_spaces_phase_a = {
    "Regresion_Logistica": {
        "model__C": [
            0.001, 0.003, 0.01, 0.03, 0.1, 0.3,
            1.0, 3.0, 10.0, 30.0, 100.0
        ],
        "model__class_weight": [None, "balanced"],
    },

    "Random_Forest": {
        "n_estimators": [300, 500, 700, 1000],
        "max_depth": [None, 8, 12, 16, 20, 24, 32],
        "min_samples_split": [2, 4, 6, 10],
        "min_samples_leaf": [1, 2, 3, 4, 6, 8],
        "max_features": ["sqrt", "log2", 0.25, 0.4, 0.6, 0.8],
        "class_weight": [None, "balanced", "balanced_subsample"],
        "bootstrap": [True],
    },

    "XGBoost": {
        "n_estimators": [200, 300, 450, 600, 800],
        "max_depth": [2, 3, 4, 5, 6, 8],
        "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
        "subsample": [0.65, 0.75, 0.85, 1.0],
        "colsample_bytree": [0.55, 0.7, 0.85, 1.0],
        "min_child_weight": [1, 2, 3, 5, 8],
        "gamma": [0.0, 0.05, 0.1, 0.25, 0.5, 1.0],
        "reg_alpha": [0.0, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0],
        "reg_lambda": [0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    },

    "SVM_RBF": {
        "model__C": [
            0.03, 0.05, 0.1, 0.2, 0.3, 0.5,
            1.0, 2.0, 3.0, 5.0, 10.0, 20.0,
            50.0, 100.0
        ],
        "model__gamma": [
            "scale",
            0.0003, 0.0005, 0.001, 0.002, 0.003,
            0.005, 0.01, 0.02, 0.03, 0.05, 0.1
        ],
        "model__class_weight": [None, "balanced"],
    },

    "MLP": {
        "model__hidden_layer_sizes": [
            (64,),
            (96,),
            (128,),
            (128, 64),
            (192, 96),
            (128, 64, 32),
            (192, 96, 48),
            (256, 128, 64),
        ],
        "model__activation": ["relu", "tanh"],
        "model__alpha": [
            0.00001, 0.0001, 0.0003, 0.001,
            0.003, 0.01, 0.03, 0.1
        ],
        "model__learning_rate_init": [
            0.0001, 0.0003, 0.0005,
            0.001, 0.002, 0.003
        ],
        "model__batch_size": [32, 64, 128],
        "model__validation_fraction": [0.12, 0.15, 0.20],
        "model__n_iter_no_change": [10, 15, 20],
    },
}

PHASE_A_BUDGETS = {
    "Regresion_Logistica": 12,
    "Random_Forest": 30,
    "XGBoost": 30,
    "SVM_RBF": 20,
    "MLP": 20,
}

print("Configuraciones Fase A:", sum(PHASE_A_BUDGETS.values()))
print(
    "Entrenamientos aproximados:",
    sum(PHASE_A_BUDGETS.values()) * len(fold_data),
)

Configuraciones Fase A: 112
Entrenamientos aproximados: 448


## 10. Motor de búsqueda temporal

In [11]:
def temporal_random_search(
    model_name,
    estimator,
    param_space,
    folds,
    n_iter,
    random_state,
    phase,
):
    sampled_params = list(
        ParameterSampler(
            param_space,
            n_iter=n_iter,
            random_state=random_state,
        )
    )

    config_rows = []
    fold_rows = []

    for config_id, params in enumerate(sampled_params, start=1):
        started = time.perf_counter()
        macro_scores = []
        balanced_scores = []
        alta_recalls = []

        for fold in folds:
            fit_started = time.perf_counter()

            candidate = clone(estimator)
            candidate.set_params(**params)
            candidate.fit(
                fold["X_train"],
                fold["y_train"],
            )

            pred = candidate.predict(fold["X_val"])
            metrics = calculate_metrics(
                fold["y_val"],
                pred,
            )

            macro_scores.append(metrics["macro_f1"])
            balanced_scores.append(metrics["balanced_accuracy"])
            alta_recalls.append(metrics["recall_alta"])

            fold_rows.append({
                "fase": phase,
                "modelo": model_name,
                "config_id": config_id,
                "fold": fold["name"],
                "fit_seconds": time.perf_counter() - fit_started,
                **metrics,
                "params": json.dumps(params, default=str),
            })

        config_rows.append({
            "fase": phase,
            "modelo": model_name,
            "config_id": config_id,
            "macro_f1_cv_mean": float(np.mean(macro_scores)),
            "macro_f1_cv_std": float(np.std(macro_scores)),
            "balanced_accuracy_cv_mean": float(
                np.mean(balanced_scores)
            ),
            "recall_alta_cv_mean": float(
                np.mean(alta_recalls)
            ),
            "seconds": time.perf_counter() - started,
            "params": params,
        })

        current = config_rows[-1]
        print(
            f"{phase} | {model_name:<20} | "
            f"{config_id:02d}/{len(sampled_params):02d} | "
            f"Macro F1={current['macro_f1_cv_mean']:.4f} "
            f"± {current['macro_f1_cv_std']:.4f} | "
            f"Recall Alta={current['recall_alta_cv_mean']:.4f} | "
            f"{current['seconds']:.1f}s"
        )

    configs_df = (
        pd.DataFrame(config_rows)
        .sort_values(
            ["macro_f1_cv_mean", "macro_f1_cv_std"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    folds_df = pd.DataFrame(fold_rows)

    return configs_df, folds_df

## 11. Fase A · búsqueda amplia de los cinco modelos

Esta es la parte pesada. Dependiendo del equipo puede tardar varios minutos.
No cierres el kernel a mitad de la fiesta.

In [12]:
phase_a_results = {}
phase_a_fold_results = []

for offset, (name, estimator) in enumerate(models.items()):
    print("\n" + "=" * 100)
    print("FASE A:", name)

    configs_df, folds_df = temporal_random_search(
        model_name=name,
        estimator=estimator,
        param_space=param_spaces_phase_a[name],
        folds=fold_data,
        n_iter=PHASE_A_BUDGETS[name],
        random_state=RANDOM_STATE + offset,
        phase="A",
    )

    phase_a_results[name] = configs_df
    phase_a_fold_results.append(folds_df)

phase_a_all = pd.concat(
    phase_a_results.values(),
    ignore_index=True,
)

phase_a_folds_all = pd.concat(
    phase_a_fold_results,
    ignore_index=True,
)

phase_a_all.assign(
    params=phase_a_all["params"].apply(
        lambda p: json.dumps(p, default=str)
    )
).to_csv(
    OUTPUT_DIR / "busqueda_fase_a.csv",
    index=False,
)

phase_a_folds_all.to_csv(
    OUTPUT_DIR / "resultados_folds_fase_a.csv",
    index=False,
)

print("\nFase A guardada.")


FASE A: Regresion_Logistica
A | Regresion_Logistica  | 01/12 | Macro F1=0.3452 ± 0.0126 | Recall Alta=0.4208 | 0.2s
A | Regresion_Logistica  | 02/12 | Macro F1=0.3357 ± 0.0125 | Recall Alta=0.3568 | 1.4s
A | Regresion_Logistica  | 03/12 | Macro F1=0.3267 ± 0.0245 | Recall Alta=0.3698 | 0.6s
A | Regresion_Logistica  | 04/12 | Macro F1=0.3524 ± 0.0079 | Recall Alta=0.4105 | 0.2s
A | Regresion_Logistica  | 05/12 | Macro F1=0.3339 ± 0.0129 | Recall Alta=0.3556 | 2.0s
A | Regresion_Logistica  | 06/12 | Macro F1=0.3328 ± 0.0144 | Recall Alta=0.3719 | 0.3s
A | Regresion_Logistica  | 07/12 | Macro F1=0.3365 ± 0.0193 | Recall Alta=0.3594 | 4.9s
A | Regresion_Logistica  | 08/12 | Macro F1=0.3327 ± 0.0162 | Recall Alta=0.3625 | 0.8s
A | Regresion_Logistica  | 09/12 | Macro F1=0.3416 ± 0.0186 | Recall Alta=0.3870 | 0.2s
A | Regresion_Logistica  | 10/12 | Macro F1=0.3310 ± 0.0212 | Recall Alta=0.3766 | 0.3s
A | Regresion_Logistica  | 11/12 | Macro F1=0.3327 ± 0.0112 | Recall Alta=0.3587 | 2.5s
A |

## 12. Ranking Fase A

In [13]:
phase_a_best_rows = []

for name, result in phase_a_results.items():
    best = result.iloc[0]

    phase_a_best_rows.append({
        "modelo": name,
        "macro_f1_cv_mean": best["macro_f1_cv_mean"],
        "macro_f1_cv_std": best["macro_f1_cv_std"],
        "balanced_accuracy_cv_mean": best[
            "balanced_accuracy_cv_mean"
        ],
        "recall_alta_cv_mean": best[
            "recall_alta_cv_mean"
        ],
        "params": best["params"],
    })

phase_a_ranking = (
    pd.DataFrame(phase_a_best_rows)
    .sort_values("macro_f1_cv_mean", ascending=False)
    .reset_index(drop=True)
)

display(
    phase_a_ranking[
        [
            "modelo",
            "macro_f1_cv_mean",
            "macro_f1_cv_std",
            "balanced_accuracy_cv_mean",
            "recall_alta_cv_mean",
        ]
    ]
)

print("\nPersistencia:")
display(
    baseline_cv[
        baseline_cv["modelo"] == "Persistencia"
    ]
)

,modelo,macro_f1_cv_mean,macro_f1_cv_std,balanced_accuracy_cv_mean,recall_alta_cv_mean
0,MLP,0.364734,0.014346,0.367090,0.354574
1,XGBoost,0.359104,0.028524,0.360921,0.361089
2,SVM_RBF,0.355663,0.019487,0.358760,0.308811
3,Regresion_Logistica,0.352436,0.007904,0.362116,0.410538
4,Random_Forest,0.348398,0.024105,0.351539,0.315914



Persistencia:


,modelo,macro_f1_cv_mean,macro_f1_cv_std,balanced_accuracy_cv_mean,recall_alta_cv_mean
1,Persistencia,0.369561,0.031969,0.369602,0.349616


## 13. Construcción automática de espacios locales · Fase B

Los dos mejores modelos ML reciben una segunda búsqueda alrededor de la mejor
configuración encontrada en Fase A.

In [14]:
def unique_sorted(values):
    cleaned = []
    for value in values:
        if value not in cleaned:
            cleaned.append(value)
    return cleaned


def refine_space(model_name, best):
    if model_name == "Regresion_Logistica":
        c = float(best["model__C"])
        return {
            "model__C": unique_sorted([
                max(c / 10, 1e-5),
                max(c / 3, 1e-5),
                c,
                c * 3,
                c * 10,
            ]),
            "model__class_weight": [None, "balanced"],
        }

    if model_name == "Random_Forest":
        n = int(best["n_estimators"])
        d = best["max_depth"]
        split = int(best["min_samples_split"])
        leaf = int(best["min_samples_leaf"])

        if d is None:
            depths = [None, 12, 16, 20, 24, 32, 40]
        else:
            depths = unique_sorted([
                None,
                max(4, d - 8),
                max(4, d - 4),
                d,
                d + 4,
                d + 8,
            ])

        return {
            "n_estimators": unique_sorted([
                max(200, int(n * 0.7)),
                n,
                int(n * 1.25),
                int(n * 1.5),
            ]),
            "max_depth": depths,
            "min_samples_split": unique_sorted([
                max(2, split - 2),
                split,
                split + 2,
                split + 4,
            ]),
            "min_samples_leaf": unique_sorted([
                max(1, leaf - 2),
                max(1, leaf - 1),
                leaf,
                leaf + 1,
                leaf + 2,
            ]),
            "max_features": unique_sorted([
                best["max_features"],
                "sqrt",
                "log2",
                0.25,
                0.4,
                0.6,
                0.8,
            ]),
            "class_weight": unique_sorted([
                best["class_weight"],
                None,
                "balanced",
                "balanced_subsample",
            ]),
            "bootstrap": [True],
        }

    if model_name == "XGBoost":
        n = int(best["n_estimators"])
        d = int(best["max_depth"])
        lr = float(best["learning_rate"])
        child = float(best["min_child_weight"])
        gamma = float(best["gamma"])
        alpha = float(best["reg_alpha"])
        lamb = float(best["reg_lambda"])

        return {
            "n_estimators": unique_sorted([
                max(100, int(n * 0.7)),
                n,
                int(n * 1.25),
                int(n * 1.5),
            ]),
            "max_depth": unique_sorted([
                max(2, d - 2),
                max(2, d - 1),
                d,
                d + 1,
                d + 2,
            ]),
            "learning_rate": unique_sorted([
                max(0.003, lr * 0.5),
                max(0.003, lr * 0.75),
                lr,
                min(0.2, lr * 1.25),
                min(0.2, lr * 1.5),
            ]),
            "subsample": unique_sorted([
                best["subsample"],
                0.65, 0.75, 0.85, 0.95, 1.0
            ]),
            "colsample_bytree": unique_sorted([
                best["colsample_bytree"],
                0.55, 0.7, 0.85, 0.95, 1.0
            ]),
            "min_child_weight": unique_sorted([
                max(1, child - 2),
                max(1, child - 1),
                child,
                child + 1,
                child + 3,
            ]),
            "gamma": unique_sorted([
                max(0.0, gamma * 0.25),
                max(0.0, gamma * 0.5),
                gamma,
                gamma * 1.5 + 0.01,
                gamma * 2 + 0.05,
            ]),
            "reg_alpha": unique_sorted([
                0.0,
                max(0.0, alpha * 0.1),
                max(0.0, alpha * 0.5),
                alpha,
                alpha * 2 + 0.001,
                alpha * 5 + 0.01,
            ]),
            "reg_lambda": unique_sorted([
                max(0.01, lamb * 0.2),
                max(0.01, lamb * 0.5),
                lamb,
                lamb * 2,
                lamb * 5,
            ]),
        }

    if model_name == "SVM_RBF":
        c = float(best["model__C"])
        gamma = best["model__gamma"]

        gamma_values = [
            "scale",
            0.0003, 0.0005, 0.001,
            0.002, 0.003, 0.005,
            0.01, 0.02, 0.03, 0.05,
        ]

        if isinstance(gamma, (int, float)):
            gamma_values = unique_sorted([
                "scale",
                max(1e-5, gamma / 5),
                max(1e-5, gamma / 2),
                gamma,
                gamma * 2,
                gamma * 5,
            ] + gamma_values)

        return {
            "model__C": unique_sorted([
                max(0.001, c / 5),
                max(0.001, c / 2),
                c,
                c * 2,
                c * 5,
            ]),
            "model__gamma": gamma_values,
            "model__class_weight": [None, "balanced"],
        }

    if model_name == "MLP":
        alpha = float(best["model__alpha"])
        lr = float(best["model__learning_rate_init"])
        hidden = best["model__hidden_layer_sizes"]

        architectures = unique_sorted([
            hidden,
            (64,),
            (96,),
            (128,),
            (128, 64),
            (192, 96),
            (128, 64, 32),
            (192, 96, 48),
            (256, 128, 64),
        ])

        return {
            "model__hidden_layer_sizes": architectures,
            "model__activation": unique_sorted([
                best["model__activation"],
                "relu",
                "tanh",
            ]),
            "model__alpha": unique_sorted([
                max(1e-6, alpha / 10),
                max(1e-6, alpha / 3),
                alpha,
                alpha * 3,
                alpha * 10,
            ]),
            "model__learning_rate_init": unique_sorted([
                max(1e-5, lr / 3),
                max(1e-5, lr / 2),
                lr,
                min(0.01, lr * 1.5),
                min(0.01, lr * 3),
            ]),
            "model__batch_size": unique_sorted([
                best["model__batch_size"],
                32,
                64,
                128,
            ]),
            "model__validation_fraction": unique_sorted([
                best["model__validation_fraction"],
                0.12,
                0.15,
                0.20,
            ]),
            "model__n_iter_no_change": unique_sorted([
                best["model__n_iter_no_change"],
                10,
                15,
                20,
                25,
            ]),
        }

    raise ValueError(model_name)


TOP_K = 2
PHASE_B_ITER = 25

top_models = phase_a_ranking.head(TOP_K)["modelo"].tolist()

print("Modelos que pasan a Fase B:", top_models)

refined_spaces = {}

for model_name in top_models:
    best_params = phase_a_results[model_name].iloc[0]["params"]
    refined_spaces[model_name] = refine_space(
        model_name,
        best_params,
    )

    print(
        f"{model_name}: espacio local listo."
    )

Modelos que pasan a Fase B: ['MLP', 'XGBoost']
MLP: espacio local listo.
XGBoost: espacio local listo.


## 14. Fase B · refinamiento de los dos mejores

In [15]:
phase_b_results = {}
phase_b_fold_results = []

for offset, model_name in enumerate(top_models):
    print("\n" + "=" * 100)
    print("FASE B:", model_name)

    configs_df, folds_df = temporal_random_search(
        model_name=model_name,
        estimator=models[model_name],
        param_space=refined_spaces[model_name],
        folds=fold_data,
        n_iter=PHASE_B_ITER,
        random_state=RANDOM_STATE + 100 + offset,
        phase="B",
    )

    phase_b_results[model_name] = configs_df
    phase_b_fold_results.append(folds_df)

phase_b_all = pd.concat(
    phase_b_results.values(),
    ignore_index=True,
)

phase_b_folds_all = pd.concat(
    phase_b_fold_results,
    ignore_index=True,
)

phase_b_all.assign(
    params=phase_b_all["params"].apply(
        lambda p: json.dumps(p, default=str)
    )
).to_csv(
    OUTPUT_DIR / "busqueda_fase_b.csv",
    index=False,
)

phase_b_folds_all.to_csv(
    OUTPUT_DIR / "resultados_folds_fase_b.csv",
    index=False,
)

print("\nFase B guardada.")


FASE B: MLP
B | MLP                  | 01/25 | Macro F1=0.3351 ± 0.0218 | Recall Alta=0.3662 | 4.8s
B | MLP                  | 02/25 | Macro F1=0.3412 ± 0.0279 | Recall Alta=0.3548 | 3.1s
B | MLP                  | 03/25 | Macro F1=0.3289 ± 0.0362 | Recall Alta=0.4194 | 7.9s
B | MLP                  | 04/25 | Macro F1=0.3472 ± 0.0171 | Recall Alta=0.3717 | 3.0s
B | MLP                  | 05/25 | Macro F1=0.3537 ± 0.0062 | Recall Alta=0.3737 | 15.9s
B | MLP                  | 06/25 | Macro F1=0.3062 ± 0.0351 | Recall Alta=0.2866 | 3.8s
B | MLP                  | 07/25 | Macro F1=0.3316 ± 0.0176 | Recall Alta=0.3119 | 2.2s
B | MLP                  | 08/25 | Macro F1=0.3370 ± 0.0143 | Recall Alta=0.3444 | 4.3s
B | MLP                  | 09/25 | Macro F1=0.2878 ± 0.0397 | Recall Alta=0.2439 | 2.1s
B | MLP                  | 10/25 | Macro F1=0.3347 ± 0.0223 | Recall Alta=0.3603 | 5.2s
B | MLP                  | 11/25 | Macro F1=0.3451 ± 0.0205 | Recall Alta=0.3525 | 5.8s
B | MLP           

## 15. Selección final interna

In [16]:
final_rows = []
final_params = {}

for model_name in models:
    candidates = [
        phase_a_results[model_name]
    ]

    if model_name in phase_b_results:
        candidates.append(
            phase_b_results[model_name]
        )

    combined = pd.concat(
        candidates,
        ignore_index=True,
    ).sort_values(
        ["macro_f1_cv_mean", "macro_f1_cv_std"],
        ascending=[False, True],
    )

    best = combined.iloc[0]

    final_params[model_name] = best["params"]

    final_rows.append({
        "modelo": model_name,
        "macro_f1_cv_mean": best["macro_f1_cv_mean"],
        "macro_f1_cv_std": best["macro_f1_cv_std"],
        "balanced_accuracy_cv_mean": best[
            "balanced_accuracy_cv_mean"
        ],
        "recall_alta_cv_mean": best[
            "recall_alta_cv_mean"
        ],
        "fase_mejor": best["fase"],
        "params": best["params"],
    })

final_ml = pd.DataFrame(final_rows)

baseline_display = baseline_cv.copy()
baseline_display["fase_mejor"] = "baseline"
baseline_display["params"] = None

final_ranking = (
    pd.concat(
        [
            final_ml,
            baseline_display[
                [
                    "modelo",
                    "macro_f1_cv_mean",
                    "macro_f1_cv_std",
                    "balanced_accuracy_cv_mean",
                    "recall_alta_cv_mean",
                    "fase_mejor",
                    "params",
                ]
            ],
        ],
        ignore_index=True,
    )
    .sort_values(
        "macro_f1_cv_mean",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    final_ranking[
        [
            "modelo",
            "macro_f1_cv_mean",
            "macro_f1_cv_std",
            "balanced_accuracy_cv_mean",
            "recall_alta_cv_mean",
            "fase_mejor",
        ]
    ]
)

final_ranking.assign(
    params=final_ranking["params"].apply(
        lambda p: (
            json.dumps(p, default=str)
            if isinstance(p, dict)
            else p
        )
    )
).to_csv(
    OUTPUT_DIR / "ranking_final_cv.csv",
    index=False,
)

with open(
    OUTPUT_DIR / "mejores_parametros_finales.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        final_params,
        f,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

print("\nArchivos finales guardados en:", OUTPUT_DIR)

,modelo,macro_f1_cv_mean,macro_f1_cv_std,balanced_accuracy_cv_mean,recall_alta_cv_mean,fase_mejor
0,Persistencia,0.369561,0.031969,0.369602,0.349616,baseline
1,MLP,0.364734,0.014346,0.367090,0.354574,A
2,XGBoost,0.359104,0.028524,0.360921,0.361089,A
3,SVM_RBF,0.355663,0.019487,0.358760,0.308811,A
4,Regresion_Logistica,0.352436,0.007904,0.362116,0.410538,A
5,Random_Forest,0.348398,0.024105,0.351539,0.315914,A
6,Dummy,0.155065,0.016693,0.333333,0.250000,baseline



Archivos finales guardados en: c:\Users\Kathy\Documents\GitHub\rain-threat-classifier\resultados_experimentos\v5_optimizacion_intensiva


## 16. Regla de decisión antes de abrir validación 2018–2021

No hace falta que el modelo llegue a 0.80 en CV para continuar. Lo que buscamos
ahora es determinar si una optimización seria cambia materialmente el resultado.

Compararemos el mejor modelo contra:

- **Persistencia CV ≈ 0.3696**
- mejores resultados V5 anteriores ≈ 0.351

Una mejora clara por encima de Persistencia justificaría pasar a la validación
2018–2021. Si después de toda esta búsqueda los cinco modelos siguen por debajo
o prácticamente empatados, tendremos evidencia fuerte de que el cuello de
botella no era una búsqueda insuficiente de hiperparámetros.

**No abrir todavía `prueba_temporal` ni `holdout_espacial`.**